In [1]:
%uv pip install numpy orjson

In [2]:
import logging

logging.basicConfig(
    level = logging.INFO, 
    datefmt="%Y-%m-%d %H:%M:%S", 
    format = "(%(asctime)s) - [%(levelname)s] - %(message)s"
)

from time import perf_counter, sleep
from numpy import array, dot, pi, arccos, linalg, random, float32, int32
from datetime import datetime, UTC
from orjson import dumps, OPT_SERIALIZE_NUMPY
from threading import Lock, Thread, Event
from os import cpu_count

from typing import Any, List, Dict
from numpy.typing import NDArray

In [3]:
rng = random.default_rng(seed = 137) # Initialize the permuted congruential generator.

vector_and_value = lambda: {
    "vector": (vector := rng.random(11).astype(float32)), # Alocate the vector on memmory to compute the value
    "value": 1 if max(vector) > 0.87 or sum(vector) > 6.5 else 0
}

**Perceptron**

The model’s output ($\hat{y}$) is computed as the weighted sum of the inputs ($  x  $) and the weights ($w$), plus a bias term ($b$).

$$
z = \sum_{i=1}^{n} w_i x_i + b 
\tag{1}
$$

This value $z$ is then passed through an activation function to produce the final prediction ($\hat{y} = f(z)$). The classic perceptron uses the step function (also called the threshold function), where 1 represents the positive class (True) and 0 represents the negative class (False).

$$
\hat{y} = f(z) = \begin{cases} 1 & \text{if } z \geq 0 \\ 0 & \text{if } z < 0 \end{cases} 
\tag{2}
$$

The perceptron learns from data by iteratively adjusting its weights and bias using the following supervised update rule. For each training example with input vector $x$, true label $y \in \{0, 1\}$, and predicted output $\hat{y}$.

If the prediction is wrong ($\hat{y} \neq y$), update the parameters as follows.

$$
w_i \leftarrow w + \eta (y - \hat{y}) x
\tag{3}
$$

$$
b \leftarrow b + \eta (y - \hat{y})
\tag{4}
$$

Where $\eta$ (eta) is the learning rate — a small positive constant that controls how much the weights are adjusted (typically $0 < \eta \leq 1$).

This update is applied to every training example, usually over multiple passes through the dataset (called epochs), until the model correctly classifies all examples or a stopping condition is reached.

In [4]:
class SimplePerceptron:
    def __init__(self) -> None:

        """
        Iniitialize the simple perceptron model with weights and bias.
        """

        self.weights: NDArray[float32] = None
        self.bias: float32 = None

    def train(
            self, 
            vectors_and_values: List[Dict[str, NDArray[float32] | int32]], 
            learning_rate: float32,
            epochs: int,
            early_stopping_threshold: int = 37,
            seed: int = 137
        ) -> Dict[str, list[float32] | float32 | int]:

        """
        Train a new model, using the given hiperparams.

        Args:
            vectors_and_values: List[Dict[str, NDArray[float32] | int32]] → List of dictionaries, containing a vector and a value.
            learning_rate: float32 → Learning step.
            epochs: int → Learning adjusts epochs. 
            early_stopping_threshold: int → Early stopping model threshold.
            seed: int → Weights initializatoin randomnes seed.

        Output:
            Dict[str, list[float32] | float32 | int] → Serialized model.
        """

        vector_length: int = len(vectors_and_values[0]["vector"])
        number_items: int = len(vectors_and_values)

        rng = random.default_rng(seed)

        self.weights: NDArray[float32] = rng.random(vector_length).astype(float32)
        self.bias: float32 = float32(rng.uniform(-1, 1))

        minor_error: float = None
        epochs_without_improvement: int = 0

        for e in range(epochs):

            epoch_errors: int = 0
            start: float = perf_counter()
            
            for row in vectors_and_values:

                input_vector: NDArray[float32] = row["vector"]
                true_label: int32 = row["value"]

                net_input: float32 = self._net_input(input_vector)
                pred_label: int32 = self._step_function(net_input)
    
                if pred_label != true_label:
    
                    self._learning_rule(learning_rate, true_label, pred_label, input_vector)
                    epoch_errors += 1

            error_rate: float = (epoch_errors / number_items) * 100
            
            last_epoch: int = e + 1

            if minor_error is None or error_rate < minor_error: # Log and save model when the current error rate it's minor to the last model's error.

                best_weights: NDArray[float32] = self.weights.copy() # Copy the past arrary on the best_weights
                best_bias: float32 = self.bias

                minor_error: float = error_rate
                epochs_without_improvement: int = 0

                logging.info(f"Current epoch: {e + 1}/{epochs}, where the duration was {perf_counter() - start:.6f} seconds and the error rate {error_rate:.2f} porcent.")

            else:
    
                epochs_without_improvement += 1
            
                if epochs_without_improvement >= early_stopping_threshold: # Early stopping mechanism.

                    last_epoch: int = e + 1
    
                    break

        self.weights: NDArray[float32] = best_weights.copy()
        self.bias: float32 = best_bias
        
        return self._serialize_model(learning_rate, last_epoch)

    def inference(self, values: NDArray[float32]) -> int32:

        """
        Make inference using the trained model on memmory

        Args:
            values: NDArray[float32] → Input vector values (x).

        Output:
            int32 → 1 if z >= 0, 0 otherwise.
        """

        if self.weights is None or self.bias is None:

            logging.error("No model loaded on the object memmory.")
            
            return None

        pred_label: float32 = self._net_input(values)

        return self._step_function(pred_label)

    def _net_input(self, values: NDArray[float32]) -> float32:

        """
        Net input formula for the model

        Args:
            values: NDArray[float32] → Input vector values (x).

        Output:
            float32 → The weighted sum of the inputs and the weights, plus a bias term.
        
        Maths:
            w * x + b
        """

        return dot(self.weights, values) + self.bias

    def _step_function(self, pred_label: float32) -> int32:

        """
        Step function for the perceptron model.

        Args:
            pred_label: float32 → The net input value (z).

        Output:
            int32 → 1 if z >= 0, 0 otherwise.

        Maths:
            1 if z >= 0, else 0
        """

        return 1 if pred_label >= 0 else 0

    def _learning_rule(self, learning_rate: float32, true_label: float32, pred_label: float32, input_vector: float32) -> None:

        """
        Learning rule for the simple perceptron.

        Args:
            learning_rate: float32 → Learning step.
            true_label: float32 → True label value.
            pred_label: float32 → Label predicted by model.
            input_vector: float32 → Input vector values (x).

        Maths:
            b ← b + η · (y - ŷ)
            w_1 ← w_1 + η · (y - ŷ) x_1
        """
        
        error: float32 = float32(true_label - pred_label)
        
        self.bias += learning_rate * error
        self.weights += learning_rate * error * input_vector

    def _serialize_model(self, learning_rate: float32, epochs: int32) -> Dict[str, list[float32] | int | float32]:

        """
        Generates a dict whit the information of the trained model.
        """
        
        return {"weights": self.weights, "bias": self.bias, "learning_rate": learning_rate, "epochs": epochs}

rows_amount: int = 100_000 # E.g., 11 * 4 = 44 bytes * 100_000 = 4.4 megabytes.

vectors_and_values: List[Dict[str, NDArray[float32] | int32]] = []

for _ in range(rows_amount):

    vectors_and_values.append(vector_and_value())

simple_perceptron = SimplePerceptron()
serialized_model: Dict[str, list[float32] | float32| int] = simple_perceptron.train(vectors_and_values, learning_rate = float32(0.01), epochs = 1028)

logging.info(f"Serialized model: {str(serialized_model)}")

**Cosine Similarity**


Cosine similarity allows us to know the similarity to determine two data points based on the angle they point toward rather than their length or magnitude. It is typically employed in hyperplanes where a distance in 11 dimensions may not behave the same as in 3 or 20 dimensions (i.e., unit cosine implies coincidence, zero indicates orthogonality, and negative one signifies diametrical opposition.).

$$
\text{similarity}(A, B) = \cos(\theta) = \frac{A \cdot B}{\|A\| \|B\|}
\tag{5}
$$

*Where for the perceptron $A$ it represents the weights vector (w) and $B$ the values vector (x).*

In [5]:
def uncertainty_head(self, values: NDArray[float32]) -> Dict[str, str | float32]:

    """
    Compute the uncertaainty of the model, using the cosine similarity.

    Args:
        values: NDArray[float32] → Input vector values (x).

    Output:
        interpreter: Dict[str, str | float32] → Interpreted results.

    Maths:
        (A · B) / (||A|| * ||B||)
    """

    dot_product: float32 = dot(self.weights, values) # Calculates alignment between the two vectors..

    # Calculates vectors' size or magnitude.
    weights_magnitude: float32 = linalg.norm(self.weights)
    values_magnitude: float32 = linalg.norm(values)

    cosine_similarity: float32 = dot_product / (weights_magnitude * values_magnitude) # Normalizes alignment into cosine similarity.

    interpreter: Dict[str, float32] = {
        "angle": arccos(cosine_similarity) * (180 / pi), # Where 0° are identical, and 180° are completely opposite.
        "value": cosine_similarity
    }

    return interpreter

SimplePerceptron.uncertainty_head = uncertainty_head # Assign the new method to the object. 

In [6]:
out_distribuation_vector: NDArray[float32] = rng.uniform(low = -600, high = -300, size = len(simple_perceptron.weights)).astype(float32) # Out of distribuation weights (dispersed and different scale).

interpreter: Dict[str, str | float32] = simple_perceptron.uncertainty_head(out_distribuation_vector)

logging.info(f"Angle of both vectors it's {interpreter["angle"]}, where the cosine similarity presents {interpreter["value"]}.")

In [7]:
class LoadGenerator:

    def __init__(self, simple_perceptron: SimplePerceptron, anomalies_ratio: float32 = 0.15) -> None:
        """
        Initialize the load generator object.

        Args:
            simple_perceptron: SimplePerceptron → Simple perceptorn object pre-trained.
            anomalies_ratio: float32 → Ratio of anomalies, where 0 its 0% and 1.0 equivalent to 100%.
        """

        self.anomalies_ratio: float32 = anomalies_ratio
        self.vector_length: int32 = len(simple_perceptron.weights)

        self.simple_perceptron: SimplePerceptron = simple_perceptron

        self.logger_lock = Lock()
        self.logs_filename: str = f"workers_{datetime.now(UTC).strftime('%Y%m%dT%H%M%S')}.jsonl"

        self._logger_system({
            "timestamp": datetime.now(UTC).isoformat(), 
            "weights": simple_perceptron.weights, 
            "bias": simple_perceptron.bias
        })

        self.stop_workers = Event() # Start on the False value.

    def worker(self, worker_seed: float32 = 137) -> None:

        """
        Worker loop method, that generate load to the simple perceptorn.

        Args:
            worker_seed: float32 → Random numbers seed.
        """

        rng: random.Generator = random.default_rng(seed = worker_seed) # Permuted congruential generator for each worker.

        while not self.stop_workers.is_set():
        
            if rng.random() <= self.anomalies_ratio: # Generate a float on the range of 0.0 and 1.0.
    
                payload: Dict[str, NDArray[float32] | int32] = self._anomalies(rng)
    
            else:
    
                payload: Dict[str, NDArray[float32] | int32] = self._sample(rng)
    
            vector: NDArray[float32] = payload["vector"]
            value: int32 | float32 = payload["value"]
    
            output: int32 = self.simple_perceptron.inference(vector)
            uncertainty: Dict[str, str | float32] = self.simple_perceptron.uncertainty_head(vector)
    
            serialized_worker_log: Dict[str, Any] = {
                "timestamp": datetime.now(UTC).isoformat(),
                "vector": vector,
                "real_value": value,
                "model_output": output,
                "vectors_angle": uncertainty["angle"],
                "cosine_similarity": uncertainty["value"]
            }

            sleep(0.001)
    
            with self.logger_lock:
        
                self._logger_system(serialized_worker_log)

    def _logger_system(self, jsonl_message: Dict[str, Any]) -> None:

        """
        Load generator logging system on binary.

        Args:
            jsonl_message: Dict[str, Any] → JSONL message to insert on the worker logger.
        """

        with open(self.logs_filename, "ab") as workers_logger:

            workers_logger.write(dumps(jsonl_message, option = OPT_SERIALIZE_NUMPY) + b"\n")

    def _sample(self, rng: random.Generator) -> Dict[str, NDArray[float32] | int32]:

        """
        Regular sample generator.

        Args:
            rng: random.Generator → Random numbers generator object.

        Output:
            Dict[str, NDArray[float32] | int32] → Serialized dict containing the vector and value.
        """

        return {
            "vector": (vector := rng.random(self.vector_length).astype(float32)),
            "value": 1 if max(vector) > 0.87 or sum(vector) > 6.5 else 0
        }

    def _anomalies(self, rng: random.Generator) -> Dict[str, NDArray[float32] | int32]:

        """
        Anomalies generator.

        Args:
            rng: random.Generator → Random numbers generator object.

        Output:
            Dict[str, NDArray[float32] | int32] → Serialized dict containing the vector and value.
        """

        if rng.random() > 0.5:
            
            transformation: float32 = rng.uniform(1, 2.35) # Positive.
        
        else:
        
            transformation: float32 = rng.uniform(-3, -1) # Negative.

        return {
            "vector": (rng.random(self.vector_length).astype(float32) ** 0.03) * transformation,
            "value": -1
        }

In [8]:
generator = LoadGenerator(simple_perceptron, 0.13)

cpus: int32 = cpu_count() or 1

for i in range(cpus):

    s: float32 = i + 137

    t = Thread(target = lambda seed = s: generator.worker(seed), daemon = True)
    t.start()

    logging.info(f"Worker number {i + 1} started with seed {s}.")

sleep(3 * 60) # Wait 3 minutes.

generator.stop_workers.set() # Set stop workers to the True value.

logging.info("Stoping workers, terminating program.")